# Image Generation Using Google Vertex AI Imagen

**Author:** [Your Name]  
**Date:** April 2, 2026  
**Course:** [Your Course]

## Overview
This notebook demonstrates the use of **Google's Imagen** model through **Vertex AI** for programmatic image generation. Imagen is Google's advanced text-to-image generation model, available through Google Cloud's Vertex AI platform.

Imagen is known for its high-quality, photorealistic image generation and strong prompt understanding.

## Setup and Installation

First, we need to install the required libraries to interact with Google Vertex AI.

**Requirements:**
1. **Google Cloud Account** - Sign up at https://cloud.google.com/
2. **Vertex AI API Enabled** - Enable in Google Cloud Console
3. **Service Account Key** - JSON key file for authentication
4. **Google Cloud SDK** - For authentication

**Available Models:**
- **imagen-3.0-generate-001** - Latest Imagen 3 model
- **imagen-3.0-fast-generate-001** - Faster version of Imagen 3
- **imagen-2.0-generate-001** - Previous generation

**Note:** Vertex AI has a free tier, but Imagen requires billing to be enabled.

In [1]:
# Install required packages
%pip install google-cloud-aiplatform pillow requests python-dotenv matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.3 MB/s eta 0:00:00 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 3.5 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 5.5 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.3/262.3 kB 6.8 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 5.9 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 5.3 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 5.9 MB/s eta 0:00:00m eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# Import required libraries
import os
import vertexai
from vertexai.preview.vision_models import ImageGenerationModel
from PIL import Image
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import json

# Load environment variables (API keys)
load_dotenv()

print("Libraries imported successfully!")

Libraries imported successfully!


## Configure Google Vertex AI

Set up your Google Cloud credentials. You have two authentication options:

**Option 1: Service Account Key (Recommended)**
- Download JSON key from Google Cloud Console
- Set GOOGLE_APPLICATION_CREDENTIALS environment variable

**Option 2: Application Default Credentials**
- Run `gcloud auth application-default login`
- Use your user account credentials

In [5]:
# Google Cloud Configuration
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "your-project-id")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")  # or "us-west1", "europe-west1", etc.

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Choose the model
# Options:
# - "imagen-3.0-generate-001" (latest, highest quality)
# - "imagen-3.0-fast-generate-001" (faster, slightly lower quality)
# - "imagen-2.0-generate-001" (previous generation)

MODEL_NAME = "imagen-3.0-generate-001"  # Using latest Imagen 3

# Load the model
model = ImageGenerationModel.from_pretrained(MODEL_NAME)

print(f"✅ Vertex AI initialized!")
print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Model: {MODEL_NAME}")

GoogleAuthError: 
Unable to authenticate your request.
Depending on your runtime environment, you can complete authentication by:
- if in local JupyterLab instance: `!gcloud auth login` 
- if in Colab:
    -`from google.colab import auth`
    -`auth.authenticate_user()`
- if in service account or other: please follow guidance in https://cloud.google.com/docs/authentication

# Assistant
This error occurs because your code is trying to use Google Cloud services without proper authentication. The Vertex AI initialization is failing because your credentials aren't set up correctly.

Would you like me to provide the corrected code with authentication steps?

## Define Image Prompts

I've created four distinct prompts that will be used to test Google's Imagen capabilities. These prompts are designed to test different aspects of image generation:

1. **Landscape/Environment** - Testing natural scene generation
2. **Character/Portrait** - Testing human/creature rendering
3. **Abstract/Artistic** - Testing creative interpretation
4. **Technical/Architectural** - Testing precision and detail

In [ ]:
# Define our four prompts
prompts = {
    "prompt1_landscape": "A serene mountain lake at sunset, with snow-capped peaks reflected in crystal clear water, pine trees in the foreground, vibrant orange and purple sky, photorealistic, 8k quality",
    
    "prompt2_character": "A wise elderly wizard with a long silver beard, wearing deep blue robes embroidered with golden stars, holding a glowing staff, kind eyes, detailed fantasy art style, dramatic lighting",
    
    "prompt3_abstract": "Abstract representation of artificial intelligence: flowing neural networks made of light, geometric patterns, cyan and magenta color palette, digital art, futuristic, high contrast",
    
    "prompt4_architectural": "Modern sustainable architecture: glass and timber eco-house built into a hillside, large windows, green roof with plants, solar panels, minimalist design, architectural photography style"
}

# Display prompts
for key, prompt in prompts.items():
    print(f"\n{key}:")
    print(f"{prompt}")

## Helper Functions

These functions will help us generate and display images using Google's Imagen.

In [ ]:
def generate_image_imagen(prompt, negative_prompt=None, aspect_ratio="1:1", 
                         person_generation=None, safety_filter_level="block_some",
                         add_watermark=False):
    """
    Generate an image using Google Imagen via Vertex AI.
    
    Args:
        prompt (str): Text description of the image to generate
        negative_prompt (str): Optional negative prompt
        aspect_ratio (str): Image aspect ratio ("1:1", "4:3", "3:4", "16:9", "9:16")
        person_generation (str): Person generation policy ("allow_adult", "block_some", "block_all")
        safety_filter_level (str): Safety filter level ("block_low_and_above", "block_medium_and_above", "block_only_high", "block_some")
        add_watermark (bool): Whether to add watermark
    
    Returns:
        PIL.Image: Generated image
    """
    
    print(f"Generating image with Google Imagen...")
    print(f"Aspect Ratio: {aspect_ratio}, Safety Level: {safety_filter_level}")
    
    # Generate the image
    response = model.generate_images(
        prompt=prompt,
        negative_prompt=negative_prompt,
        number_of_images=1,
        aspect_ratio=aspect_ratio,
        person_generation=person_generation,
        safety_filter_level=safety_filter_level,
        add_watermark=add_watermark
    )
    
    # Get the image
    image = response.images[0]
    
    print(f"✅ Image generated successfully!")
    print(f"   Generation time: {response._generation_time:.2f} seconds")
    
    return image


def save_image(image, filename):
    """
    Save an image to file.
    
    Args:
        image (PIL.Image): Image to save
        filename (str): Filename to save as
    """
    image.save(filename)
    print(f"Image saved as: {filename}")


def display_image(image, title="Generated Image"):
    """
    Display an image using matplotlib.
    
    Args:
        image (PIL.Image): Image to display
        title (str): Title for the plot
    """
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.title(title, fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()


print("Helper functions defined successfully!")

## Image Generation with Google Imagen

Now we'll use Google's Imagen 3 model to generate images from our prompts.

### Parameters Explained:
- **prompt**: The text description of what we want to generate
- **negative_prompt**: Optional - what to avoid in the image
- **aspect_ratio**: Image aspect ratio (1:1, 4:3, 16:9, etc.)
- **person_generation**: Policy for generating people (allow/block)
- **safety_filter_level**: Content safety filtering level
- **add_watermark**: Whether to add Google's watermark

### Important Notes:
- Imagen 3 generates images at high resolution automatically
- Aspect ratios determine the final dimensions
- Safety filters are applied by default
- Generation typically takes 10-30 seconds

### Prompt 1: Landscape Generation

In [ ]:
# Generate Image 1: Landscape
print("=" * 80)
print("GENERATING LANDSCAPE IMAGE")
print("=" * 80)
print(f"\nPrompt: {prompts['prompt1_landscape']}")
print("\nThis may take 15-30 seconds...\n")

img1 = generate_image_imagen(
    prompt=prompts['prompt1_landscape'],
    negative_prompt="ugly, blurry, low quality, distorted, deformed, watermark, text",
    aspect_ratio="16:9",  # Wide format for landscape
    safety_filter_level="block_some"
)

# Save and display the image
save_image(img1, "landscape_imagen.png")
display_image(img1, "Prompt 1: Mountain Lake Landscape (Google Imagen)")

**Observations on Landscape Generation:**

*[After running, add your observations here about:]*
- *Quality and realism*
- *Accuracy to the prompt*
- *Color vibrancy and composition*
- *Resolution and detail level*
- *Any impressive or disappointing details*

### Prompt 2: Character Generation

In [ ]:
# Generate Image 2: Character
print("=" * 80)
print("GENERATING CHARACTER IMAGE")
print("=" * 80)
print(f"\nPrompt: {prompts['prompt2_character']}")
print("\nThis may take 15-30 seconds...\n")

img2 = generate_image_imagen(
    prompt=prompts['prompt2_character'],
    negative_prompt="ugly, distorted face, extra limbs, deformed hands, blurry, low quality, multiple people",
    aspect_ratio="1:1",
    person_generation="allow_adult",  # Allow fantasy characters
    safety_filter_level="block_some"
)

# Save and display the image
save_image(img2, "character_imagen.png")
display_image(img2, "Prompt 2: Wizard Character (Google Imagen)")

**Observations on Character Generation:**

*[After running, add your observations here about:]*
- *Facial detail and expression*
- *Costume and prop accuracy*
- *Overall character quality and coherence*
- *Fantasy art style effectiveness*
- *Hand and anatomical accuracy*

### Prompt 3: Abstract Art Generation

In [ ]:
# Generate Image 3: Abstract
print("=" * 80)
print("GENERATING ABSTRACT ART IMAGE")
print("=" * 80)
print(f"\nPrompt: {prompts['prompt3_abstract']}")
print("\nThis may take 15-30 seconds...\n")

img3 = generate_image_imagen(
    prompt=prompts['prompt3_abstract'],
    negative_prompt="realistic, photographic, blurry, low quality, dull colors",
    aspect_ratio="1:1",
    safety_filter_level="block_some"
)

# Save and display the image
save_image(img3, "abstract_imagen.png")
display_image(img3, "Prompt 3: Abstract AI Representation (Google Imagen)")

**Observations on Abstract Art Generation:**

*[After running, add your observations about:]*
- *Color accuracy (cyan and magenta palette)*
- *Creative interpretation of the concept*
- *Complexity and detail*
- *How well it represents "artificial intelligence"*
- *Artistic coherence and visual appeal*

### Prompt 4: Architectural Generation

In [ ]:
# Generate Image 4: Architecture
print("=" * 80)
print("GENERATING ARCHITECTURAL IMAGE")
print("=" * 80)
print(f"\nPrompt: {prompts['prompt4_architectural']}")
print("\nThis may take 15-30 seconds...\n")

img4 = generate_image_imagen(
    prompt=prompts['prompt4_architectural'],
    negative_prompt="ugly, distorted, unrealistic proportions, blurry, low quality, cartoon",
    aspect_ratio="4:3",  # Good for architectural shots
    safety_filter_level="block_some"
)

# Save and display the image
save_image(img4, "architecture_imagen.png")
display_image(img4, "Prompt 4: Sustainable Architecture (Google Imagen)")

**Observations on Architectural Generation:**

*[After running, add your observations about:]*
- *Structural realism and feasibility*
- *Detail accuracy (glass, timber, solar panels, etc.)*
- *Design coherence and aesthetics*
- *Integration with landscape*
- *Professional architectural quality*

## Advanced: Aspect Ratio Comparison

Let's test how different aspect ratios affect the same prompt.

In [ ]:
# Test different aspect ratios with the same prompt
test_prompt = "A modern coffee shop interior with natural light"

aspect_ratios = ["1:1", "4:3", "16:9", "3:4"]
aspect_results = []

for ratio in aspect_ratios:
    print(f"\nGenerating with aspect ratio {ratio}...")
    img = generate_image_imagen(
        prompt=test_prompt,
        negative_prompt="dark, empty, ugly",
        aspect_ratio=ratio,
        safety_filter_level="block_some"
    )
    aspect_results.append((img, ratio))
    save_image(img, f"coffee_aspect_{ratio.replace(':', 'x')}.png")

# Display all aspect ratios
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
fig.suptitle('Aspect Ratio Comparison', fontsize=16, fontweight='bold')

for idx, (img, ratio) in enumerate(aspect_results):
    row = idx // 2
    col = idx % 2
    axes[row, col].imshow(img)
    axes[row, col].axis('off')
    axes[row, col].set_title(f'Aspect Ratio: {ratio}', fontsize=12)

plt.tight_layout()
plt.show()

print("\n💡 Notice how aspect ratios change the composition:")
print("- 1:1: Square format, balanced composition")
print("- 4:3: Traditional photo format")
print("- 16:9: Wide cinematic format")
print("- 3:4: Portrait/tall format")

## Comparison: All Generated Images

In [ ]:
# Display all generated images in a grid
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
fig.suptitle('Google Imagen Generated Images Gallery', fontsize=16, fontweight='bold', y=0.98)

images = [
    (img1, "Landscape: Mountain Lake"),
    (img2, "Character: Wizard"),
    (img3, "Abstract: AI Neural Network"),
    (img4, "Architecture: Eco-house"),
]

for idx, (img, title) in enumerate(images):
    row = idx // 2
    col = idx % 2
    axes[row, col].imshow(img)
    axes[row, col].axis('off')
    axes[row, col].set_title(title, fontsize=12)

plt.tight_layout()
plt.savefig('imagen_gallery.png', dpi=150, bbox_inches='tight')
print("Gallery saved as 'imagen_gallery.png'")
plt.show()

print("\n" + "="*60)
print("IMAGE GENERATION COMPLETE")
print("="*60)
print(f"Total images generated: {len(images)}")
print("\nAll images have been saved to the current directory.")

## Analysis and Evaluation

### Overall Assessment of Google Imagen

**Strengths:**
*[Fill in after running all prompts]*
- 
- 
- 

**Weaknesses:**
*[Fill in after running all prompts]*
- 
- 
- 

**Ease of Use:**
*[Rate from 1-10 and explain]*
- Setup complexity: 
- API documentation: 
- Parameter control: 
- Error handling: 
- Authentication process: 

**Quality Assessment:**
*[Rate each category 1-10]*
- Photorealism: 
- Prompt adherence: 
- Artistic quality: 
- Technical details: 
- Consistency: 

**API Experience:**
- Generation time: 
- Reliability: 
- Cost-effectiveness: 
- Safety filtering: 
- Aspect ratio flexibility: 

**Comparison with Other Services:**
*[How does Google Imagen compare to DALL-E, Stability AI, etc.]*
- 
- 
- 

## Advanced: Safety and Content Policies

Google Imagen has built-in safety filters. Let's test how they work:

In [ ]:
# Test safety filters with different levels
test_prompts = [
    "A beautiful landscape with mountains and a lake",
    "A fantasy scene with magical creatures",
    "A modern cityscape at night"
]

safety_levels = ["block_some", "block_only_high"]

for prompt in test_prompts:
    print(f"\nTesting prompt: '{prompt}'")
    for level in safety_levels:
        try:
            print(f"  Safety level: {level}")
            img = generate_image_imagen(
                prompt=prompt,
                aspect_ratio="1:1",
                safety_filter_level=level
            )
            print(f"    ✅ Generated successfully")
            save_image(img, f"safety_test_{level}_{prompt[:20].replace(' ', '_')}.png")
        except Exception as e:
            print(f"    ❌ Failed: {str(e)}")

print("\n💡 Safety filters help ensure appropriate content generation.")
print("   Different levels provide varying degrees of filtering.")

## Conclusion

### Key Takeaways:

1. **Google Imagen (Vertex AI)**:
   - *[Your findings about Google's Imagen experience]*
   
2. **Best Practices for Prompting:**
   - *[What prompt strategies worked best]*
   
3. **Aspect Ratios:**
   - *[How aspect ratios affected composition and quality]*
   
4. **Safety Features:**
   - *[How safety filters impacted generation]*

### Overall Rating: __/10

**Would you recommend Google Imagen for:**
- Professional photography needs: ☐ Yes ☐ No
- Creative concept art: ☐ Yes ☐ No
- Marketing materials: ☐ Yes ☐ No
- Technical illustrations: ☐ Yes ☐ No
- Enterprise applications: ☐ Yes ☐ No

**Best For:**
- *[What use cases is Google Imagen ideal for?]*

## Additional Resources

- **Google Cloud Vertex AI**: https://cloud.google.com/vertex-ai
- **Imagen Documentation**: https://cloud.google.com/vertex-ai/docs/generative-ai/image/overview
- **Get Started**: https://cloud.google.com/vertex-ai/docs/generative-ai/image/generate-images
- **Pricing**: https://cloud.google.com/vertex-ai/pricing
- **Safety Guidelines**: https://cloud.google.com/vertex-ai/docs/generative-ai/image/safety-settings
- **Python SDK**: https://cloud.google.com/python/docs/reference/aiplatform/latest

**Setup Instructions:**
1. Create Google Cloud Project
2. Enable Vertex AI API
3. Create Service Account and download JSON key
4. Set GOOGLE_APPLICATION_CREDENTIALS environment variable
5. Enable billing (required for Imagen)

## Next Steps

Now that you've generated images using Google Imagen:

1. **Compare with other platforms:**
   - DALL-E 3 (via Azure OpenAI or ChatGPT)
   - Microsoft Copilot (Bing Image Creator)
   - Stability AI (official platform)
   - Replicate (wrapper service)
   - Hugging Face (community models)
   - Adobe Firefly
   - Midjourney

2. **Evaluate differences:**
   - Image quality and photorealism
   - API ease of use and setup complexity
   - Cost and pricing models
   - Generation speed
   - Safety and content policies
   - Aspect ratio flexibility
   - Enterprise features

3. **Test enterprise features:**
   - Batch generation
   - Custom fine-tuning
   - Integration with other Google Cloud services

4. **Write your comprehensive comparison paper** analyzing all platforms

Good luck with your assignment!